In [1]:
import pandas as pd
import pysam
import os
import json

# assemblies

In [2]:
assemblyDict={}
directory='/LeeLab/Assemblies/HPRC_Release2/chrY_assemblies/'
for file in os.listdir(directory):
    if '.fai' in file or '.gzi' in file or '.DS' in file:
        continue
    else:
        assemblyDict[file.split("_")[0]]= directory+file
print(len(assemblyDict))

143


In [3]:
df = pd.read_csv("/LeeLab/HPRC/chromosomeY/Data/DavidPorubsky/AZFc_ColorBLock_Clusters_12162025.csv")

In [4]:
rmDict={}
directory='/LeeLab/HPRC/chromosomeY/Data/RepeatMasker/'
for file in os.listdir(directory):
    rmDict[file.split(".")[0]]=directory+file

## Read in MSA

In [5]:
bpDF = pd.read_csv('/LeeLab/HPRC/chromosomeY/Data/Deletion_Breakpoints/Deletion_Samples_BreakpointWindows.csv')

In [6]:
contigInfo=[]
for contig in set(bpDF['deleted_contig']):
    breakDF = bpDF[bpDF['deleted_contig']==contig].copy()
    blockDF = df[df['contig']==contig].copy()

    for row in breakDF.index:
        start = int(breakDF.at[row, 'breakWindow_start_deleted'])
        end = int(breakDF.at[row, 'breakWindow_end_deleted'])

        test = blockDF[(blockDF['start']<=start) & (blockDF['end']>=end)].reset_index().copy()
        if len(test)==1:
            print(contig, start, end, test.loc[0,'colorblock'], test.loc[0,'start'], test.loc[0,'end'])
            contigInfo.append([contig+":"+str(start)+"-"+str(end), test.loc[0,'colorblock'][:-1], contig+":"+str(test.loc[0,'start'])+"-"+str(test.loc[0,'end']),str(test.loc[0,'orientation'])])

HG04228_chrY 24240742 24249711 red2 24136500 24273180
HG00329_chrY 24373672 24374922 green3 24231291 24538440
HG02129_chrY 24186434 24192217 green1 24004540 24311637
NA18952_chrY 23990863 24004932 green1 23704459 24011541
NA18952_chrY 23918028 23949080 green1 23704459 24011541
NA18952_chrY 23824545 23830690 green1 23704459 24011541
NA18952_chrY 23807187 23812758 green1 23704459 24011541
NA18952_chrY 23709047 23711298 green1 23704459 24011541
HG01099_chrY 25068511 25071500 green1 24869787 25176899
HG01099_chrY 25020026 25023035 green1 24869787 25176899
HG01099_chrY 25227936 25229874 red3 25177194 25299583
HG00290_chrY 24749476 24750726 green3 24607094 24914238
NA18747_chrY 24741838 24743088 green3 24599466 24906601
NA18983_chrY 23749028 23753408 green1 23640640 23947831
NA18983_chrY 23704296 23733098 green1 23640640 23947831
NA18983_chrY 23645228 23647481 green1 23640640 23947831
NA18983_chrY 24030939 24047069 red3 23948117 24080039
NA18983_chrY 23901257 23903454 green1 23640640 2394783

In [7]:
blockInfoDF = pd.DataFrame(data=contigInfo, columns=['breakpointWindow','color','colorBlockCoordinates','orientation'])

In [9]:
import pandas as pd

def add_relative_positions_with_orientation(
    blockInfoDF: pd.DataFrame,
    *,
    window_col: str = "breakpointWindow",
    block_col: str = "colorBlockCoordinates",
    orientation_col: str = "orientation",
    antisense_value: str = "antisense",
) -> pd.DataFrame:
    """
    Adds relative_start/end/mid (0..1) for breakpointWindow within colorBlockCoordinates,
    and also adds oriented versions where antisense blocks are flipped:
        rel_oriented = 1 - rel

    Output columns:
      block_start, block_end, block_len
      win_start, win_end, win_len, win_mid
      relative_start, relative_end, relative_mid
      relative_start_oriented, relative_end_oriented, relative_mid_oriented
      in_block
    """

    def parse_coords(coord_str: str):
        # expects contig:start-end
        coord_str = str(coord_str)
        contig, coords = coord_str.split(":", 1)
        s, e = coords.split("-", 1)
        s = int(s)
        e = int(e)
        lo, hi = (s, e) if s <= e else (e, s)
        return contig, lo, hi

    out = blockInfoDF.copy()

    cols = {
        "block_start": [],
        "block_end": [],
        "block_len": [],
        "win_start": [],
        "win_end": [],
        "win_len": [],
        "win_mid": [],
        "relative_start": [],
        "relative_end": [],
        "relative_mid": [],
        "relative_start_oriented": [],
        "relative_end_oriented": [],
        "relative_mid_oriented": [],
        "in_block": [],
    }

    for i in out.index:
        _, b_lo, b_hi = parse_coords(out.at[i, block_col])
        _, w_lo, w_hi = parse_coords(out.at[i, window_col])

        block_len = b_hi - b_lo + 1
        win_len = w_hi - w_lo + 1
        win_mid = (w_lo + w_hi) / 2.0

        in_block = (w_lo >= b_lo) and (w_hi <= b_hi)

        w_clip_lo = max(b_lo, w_lo)
        w_clip_hi = min(b_hi, w_hi)
        w_clip_mid = (w_clip_lo + w_clip_hi) / 2.0

        denom = max(1, block_len - 1)
        rel_start = (w_clip_lo - b_lo) / denom
        rel_end   = (w_clip_hi - b_lo) / denom
        rel_mid   = (w_clip_mid - b_lo) / denom

        orient = str(out.at[i, orientation_col]).strip().lower() if orientation_col in out.columns else "sense"
        if orient == antisense_value.lower():
            o_start = 1.0 - rel_end
            o_end   = 1.0 - rel_start
            o_mid   = 1.0 - rel_mid
        else:
            o_start, o_end, o_mid = rel_start, rel_end, rel_mid

        cols["block_start"].append(b_lo)
        cols["block_end"].append(b_hi)
        cols["block_len"].append(block_len)
        cols["win_start"].append(w_lo)
        cols["win_end"].append(w_hi)
        cols["win_len"].append(win_len)
        cols["win_mid"].append(win_mid)

        cols["relative_start"].append(rel_start)
        cols["relative_end"].append(rel_end)
        cols["relative_mid"].append(rel_mid)

        cols["relative_start_oriented"].append(o_start)
        cols["relative_end_oriented"].append(o_end)
        cols["relative_mid_oriented"].append(o_mid)

        cols["in_block"].append(in_block)

    for k, v in cols.items():
        out[k] = v

    return out


In [10]:
import pandas as pd
from typing import Any, Dict, List

def group_windows_by_color_orient_and_oriented_overlap(
    df: pd.DataFrame,
    *,
    color_col: str = "color",
    orientation_col: str = "orientation",         
    window_col: str = "breakpointWindow",
    rel_start_col: str = "relative_start_oriented", 
    rel_end_col: str = "relative_end_oriented",
    in_block_col: str = "in_block",
    require_in_block: bool = True,
    merge_tolerance: float = 0.01,  
) -> pd.DataFrame:
    """
    Group breakpoint windows whose oriented relative intervals overlap OR are within
    `merge_tolerance` (gap <= tolerance), but keep groups separate across orientations.

    Grouping is performed within each (color, orientation) subset.
    Oriented coordinates are still used for interval positions (antisense is flipped upstream).
    """

    out_rows: List[Dict[str, Any]] = []
    work = df.copy()

    if require_in_block and in_block_col in work.columns:
        work = work[work[in_block_col] == True].copy()

    needed = [color_col, orientation_col, window_col, rel_start_col, rel_end_col]
    missing = [c for c in needed if c not in work.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    work[rel_start_col] = pd.to_numeric(work[rel_start_col], errors="coerce")
    work[rel_end_col]   = pd.to_numeric(work[rel_end_col], errors="coerce")
    work = work.dropna(subset=needed).copy()

    work["lo"] = work[[rel_start_col, rel_end_col]].min(axis=1)
    work["hi"] = work[[rel_start_col, rel_end_col]].max(axis=1)

    def flush_group(color_val, orient_val, gid, members):
        if not members:
            return
        g_lo = min(m["lo"] for m in members)
        g_hi = max(m["hi"] for m in members)
        g_size = len(members)
        for m in members:
            out_rows.append({
                color_col: color_val,
                orientation_col: orient_val,
                "overlap_group_id": gid,    
                window_col: m["win"],
                rel_start_col: m["lo"],
                rel_end_col: m["hi"],
                "group_rel_start": g_lo,
                "group_rel_end": g_hi,
                "group_size": g_size,
            })

    for (color_val, orient_val), sub in work.groupby([color_col, orientation_col], sort=False):
        sub = sub.sort_values(["lo", "hi", window_col]).copy()

        group_id = 0
        current_end = None
        members = []

        for win, lo, hi in sub[[window_col, "lo", "hi"]].itertuples(index=False, name=None):
            if current_end is None:
                group_id += 1
                current_end = hi
                members = [{"win": win, "lo": lo, "hi": hi}]
            else:
                if lo <= current_end + merge_tolerance:
                    members.append({"win": win, "lo": lo, "hi": hi})
                    current_end = max(current_end, hi)
                else:
                    flush_group(color_val, orient_val, group_id, members)
                    group_id += 1
                    current_end = hi
                    members = [{"win": win, "lo": lo, "hi": hi}]

        flush_group(color_val, orient_val, group_id, members)

    return pd.DataFrame(out_rows)


In [11]:
blockInfoDF2 = add_relative_positions_with_orientation(blockInfoDF)
groupsDF = group_windows_by_color_orient_and_oriented_overlap(blockInfoDF2, merge_tolerance=0.01)

In [12]:
groupsDF

,color,orientation,overlap_group_id,breakpointWindow,relative_start_oriented,relative_end_oriented,group_rel_start,group_rel_end,group_size
0,red,sense,1,HG04228_chrY:24240742-24249711,0.762672,0.828292,0.762672,0.828292,1
1,green,sense,1,HG00358_chrY:24782575-24792339,0.435849,0.467638,0.435849,0.467638,4
2,green,sense,1,NA18747_chrY:24741838-24743088,0.463549,0.467618,0.435849,0.467638,4
3,green,sense,1,HG00329_chrY:24373672-24374922,0.463557,0.467626,0.435849,0.467638,4
4,green,sense,1,HG00290_chrY:24749476-24750726,0.463568,0.467637,0.435849,0.467638,4
5,green,antisense,1,NA18952_chrY:23990863-24004932,0.021522,0.067337,0.021522,0.114435,2
6,green,antisense,1,HG005_chrY:24314836-24334066,0.051824,0.114435,0.021522,0.114435,2
7,green,antisense,2,NA18983_chrY:23901257-23903454,0.144461,0.151613,0.144461,0.151613,1
8,green,antisense,3,NA18952_chrY:23918028-23949080,0.203402,0.304521,0.203402,0.352927,3
9,green,antisense,3,HG005_chrY:24244571-24256461,0.304498,0.343211,0.203402,0.352927,3


In [ ]:
#HG04228_chrY:24240742-24249711
#HG01099_chrY:25068511-25071500 0.343194, 0.501003, 0.569569
NA18983
HG005